# Proyecto de Investigación en Inteligencia Artificial (FIEI - UNFV)
## Sistema Inteligente para el Diagnóstico y Tamizaje de Anemia Infantil en Población Vulnerable del Perú a partir de Microdatos de la ENDES (INEI)

**Universidad:** Universidad Nacional Federico Villarreal (UNFV)  
**Facultad:** Facultad de Ingeniería Electrónica e Informática (FIEI)  
**Asignatura:** Inteligencia Artificial / Metodología de la Investigación Científica  
**Docente:** Dr. Ciro Rodríguez Rodríguez  
**Alumno:** Jimenez Villalva Franz José  
**Año:** 2026  

---
### Propósito del Cuaderno:
Este cuaderno documenta todo el ciclo de ingeniería de datos y aprendizaje automático siguiendo la metodología CRISP-DM:
1. **Carga y Fusión de Microdatos Oficiales de la ENDES (INEI Perú)** (Módulo RECH6 de salud infantil y RECH0 de hogar).
2. **Limpieza, Control de Calidad y Corrección de Hemoglobina por Altitud** según la Norma Técnica NTS N° 134-MINSA.
3. **Análisis Exploratorio de Datos (EDA)** comparando prevalencias en regiones como Puno, Loreto y Lima.
4. **Prevención Estricta de Fuga de Datos (Data Leakage)** mediante partición estratificada 80/20 y escalado aislado.
5. **Entrenamiento y Comparación de Modelos:** K-Nearest Neighbors (KNN), Random Forest y Baselines de referencia.
6. **Interpretabilidad (Gini Importance)** demostrando el impacto de la altitud y la hemoglobina en el diagnóstico peruano.
7. **Prototipo de Inferencia Clínica** para tamizaje en tiempo real en postas de salud del país.

## 1. Instalación e Importación de Librerías

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Si se ejecuta en Google Colab, clonar el repositorio si no existe para tener acceso a los microdatos
if 'google.colab' in sys.modules:
    if not os.path.exists('data'):
        print(':: Entorno Google Colab detectado. Clonando repositorio oficial...')
        !git clone https://github.com/MAKIZAPA/proyecto-ia-anemia.git
        %cd proyecto-ia-anemia
        print(':: Repositorio listo.')

# Modelado y métricas con Scikit-Learn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)
import joblib

# Estilo visual sobrio
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print(':: Librerías y entorno inicializados correctamente.')

## 2. Ingesta, Fusión y Limpieza de Microdatos Oficiales de la ENDES (INEI Perú)
Para que todo el trabajo sea claro y verificable, aquí mostramos cómo cargamos y unimos directamente los archivos originales del INEI:
- **RECH6:** Módulo de salud infantil (peso, talla y la hemoglobina medida en campo).
- **RECH0:** Módulo del hogar (departamento, altitud en msnm y si la zona es urbana o rural).

En el siguiente bloque hacemos match con el código del hogar (`HHID`), descartamos mediciones con errores o fuera de rango normal, y aplicamos la fórmula oficial del **MINSA (NTS N° 134)** para ajustar la hemoglobina por altitud.

In [ ]:
# Rutas a los datos brutos del INEI
raw_r6 = 'data/raw/RECH6_2023.csv'
raw_r0 = 'data/raw/RECH0_2023.csv'

# Si estamos en Colab y faltara algún archivo, se descarga al instante desde el repositorio
if not (os.path.exists(raw_r6) and os.path.exists(raw_r0)):
    print(':: Descargando microdatos oficiales desde GitHub...')
    import urllib.request
    os.makedirs('data/raw', exist_ok=True)
    base_raw = 'https://raw.githubusercontent.com/MAKIZAPA/proyecto-ia-anemia/main/data/raw/'
    for fname in ['RECH6_2023.csv', 'RECH0_2023.csv']:
        urllib.request.urlretrieve(base_raw + fname, f'data/raw/{fname}')
    print(':: Descarga lista.')

print(':: [1/4] Leyendo microdatos crudos del INEI...')
r6 = pd.read_csv(raw_r6, low_memory=False)
r0 = pd.read_csv(raw_r0, low_memory=False)

r6.columns = [c.strip().replace('ï»¿', '') for c in r6.columns]
r0.columns = [c.strip().replace('ï»¿', '') for c in r0.columns]
print(f' -> Registros en RECH6 (Niños): {r6.shape[0]}')
print(f' -> Registros en RECH0 (Hogar): {r0.shape[0]}')

print(':: [2/4] Cruzando datos por el código del hogar (HHID)...')
r6['HHID'] = r6['HHID'].astype(str).str.strip()
r0['HHID'] = r0['HHID'].astype(str).str.strip()
merged = pd.merge(r6, r0[['HHID', 'HV040', 'HV024', 'HV025']], on='HHID', how='inner')
print(f' -> Total de casos cruzados: {merged.shape[0]}')

print(':: [3/4] Filtrando datos válidos y coherentes...')
# 1. Niños menores de 3 años (6 a 35 meses) y con prueba bien tomada (HC55 == 0)
df_clean = merged[(merged['HC55'] == 0) & (merged['HC1'] >= 6) & (merged['HC1'] <= 35)].copy()

# 2. Hemoglobina en escala real (dividida entre 10) y dentro del rango normal medible (4.0 a 20.0 g/dL)
df_clean['Hemoglobina_Observada'] = df_clean['HC53'] / 10.0
df_clean = df_clean[(df_clean['Hemoglobina_Observada'] >= 4.0) & (df_clean['Hemoglobina_Observada'] <= 20.0)].copy()

# 3. Filtro de rangos normales de peso (3 a 30 kg) y talla (45 a 120 cm) para evitar errores de tipeo
df_clean['Peso_kg'] = pd.to_numeric(df_clean['HC2'], errors='coerce') / 10.0
df_clean['Talla_cm'] = pd.to_numeric(df_clean['HC3'], errors='coerce') / 10.0
df_clean = df_clean.dropna(subset=['Peso_kg', 'Talla_cm'])
df_clean = df_clean[(df_clean['Peso_kg'] >= 3.0) & (df_clean['Peso_kg'] <= 30.0)]
df_clean = df_clean[(df_clean['Talla_cm'] >= 45.0) & (df_clean['Talla_cm'] <= 120.0)].copy()

print(':: [4/4] Aplicando factor de ajuste MINSA por altitud...')
df_clean['Altitud_msnm'] = df_clean['HV040'].astype(float)
alt_km = df_clean['Altitud_msnm'] / 1000.0

# Si está por encima de los 1000 msnm restamos el factor MINSA para corregir el oxígeno
factor_ajuste = np.where(
    df_clean['Altitud_msnm'] < 1000.0,
    0.0,
    -0.032 * alt_km + 0.022 * (alt_km ** 2)
)
df_clean['Hemoglobina_Ajustada'] = np.round(df_clean['Hemoglobina_Observada'] - factor_ajuste, 2)
# Diagnóstico oficial: menor a 11.0 g/dL es Anemia
df_clean['Anemia'] = np.where(df_clean['Hemoglobina_Ajustada'] < 11.0, 1, 0)

# Dejamos variables claras para el análisis
df_clean['Edad_Meses'] = df_clean['HC1'].astype(int)
df_clean['Sexo'] = np.where(df_clean['HC27'] == 1, 1, 0) # 1: Varón, 0: Mujer
df_clean['Area_Residencia'] = np.where(df_clean['HV025'] == 2, 1, 0) # 1: Rural, 0: Urbano

dept_map = {
    1: 'Amazonas', 2: 'Ancash', 3: 'Apurimac', 4: 'Arequipa', 5: 'Ayacucho',
    6: 'Cajamarca', 7: 'Callao', 8: 'Cusco', 9: 'Huancavelica', 10: 'Huanuco',
    11: 'Ica', 12: 'Junin', 13: 'La Libertad', 14: 'Lambayeque', 15: 'Lima',
    16: 'Loreto', 17: 'Madre de Dios', 18: 'Moquegua', 19: 'Pasco', 20: 'Piura',
    21: 'Puno', 22: 'San Martin', 23: 'Tacna', 24: 'Tumbes', 25: 'Ucayali'
}
df_clean['Departamento'] = df_clean['HV024'].map(dept_map)

cols_finales = [
    'Hemoglobina_Observada', 'Altitud_msnm', 'Edad_Meses', 'Sexo',
    'Area_Residencia', 'Peso_kg', 'Talla_cm', 'Departamento',
    'Hemoglobina_Ajustada', 'Anemia'
]
df = df_clean[cols_finales].reset_index(drop=True)
print(f'\n:: Muestra final limpia: {df.shape[0]} niños peruanos x {df.shape[1]} variables.')
df.head()

### Explicación del Proceso de Limpieza y Corrección por Altitud (MINSA / INEI):
1. **Población Objetivo:** Niños entre 6 y 35 meses (menores de 3 años, población crítica de los programas sociales).
2. **Filtro de Calidad:** Solo mediciones de hemoglobina válidas tomadas en campo (`HC55 == 0`) entre 4.0 y 20.0 g/dL.
3. **Fórmula de Corrección por Altitud (NTS N° 134-MINSA):**  
   En el Perú, la falta de oxígeno en ciudades de la sierra (como Puno, Cusco o Huancavelica) produce policitemia adaptativa (más glóbulos rojos). Para saber si un niño tiene anemia real, el MINSA exige restar un factor de corrección si la altitud supera los 1,000 msnm:  
   $$\text{alt} = \text{Altitud} / 1000$$
   $$\text{Factor} = -0.032 \times \text{alt} + 0.022 \times \text{alt}^2$$
   $$\text{Hb\_Ajustada} = \text{Hb\_Observada} - \text{Factor}$$
   $$\text{Anemia} = 1 \quad \text{si } \text{Hb\_Ajustada} < 11.0 \text{ g/dL}$$

In [ ]:
# Resumen de variables y distribución de clases en la población infantil peruana
print("--- DISTRIBUCIÓN DE ANEMIA EN MENORES DE 3 AÑOS (INEI ENDES) ---")
conteo = df['Anemia'].value_counts()
print(conteo)
print(f"Tasa de Anemia en la muestra: {df['Anemia'].mean()*100:.2f}%")
print(f"Altitud promedio: {df['Altitud_msnm'].mean():.1f} msnm (Máxima: {df['Altitud_msnm'].max()} msnm)")

## 3. Análisis Exploratorio de Datos (EDA en el Contexto Peruano)

In [ ]:
# Prevalencia por departamentos del Perú
plt.figure(figsize=(12, 5))
prevalencia_dpto = df.groupby('Departamento')['Anemia'].mean().sort_values(ascending=False) * 100
sns.barplot(x=prevalencia_dpto.index, y=prevalencia_dpto.values, palette='Blues_r')
plt.title('Tasa de Anemia Infantil por Departamento del Perú (ENDES INEI)')
plt.xlabel('Departamento')
plt.ylabel('Prevalencia (%)')
plt.xticks(rotation=45, ha='right')
plt.axhline(df['Anemia'].mean()*100, color='red', linestyle='--', label=f'Promedio Nacional ({df["Anemia"].mean()*100:.1f}%)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Relación entre Altitud y Hemoglobina Observada
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.scatterplot(data=df, x='Altitud_msnm', y='Hemoglobina_Observada', hue='Anemia', alpha=0.5, palette=['#2F5496', '#C00000'], ax=axes[0])
axes[0].set_title('Hemoglobina Observada vs Altitud (msnm)')
axes[0].set_xlabel('Altitud sobre el nivel del mar (msnm)')
axes[0].set_ylabel('Hemoglobina Observada (g/dL)')

sns.histplot(data=df, x='Hemoglobina_Ajustada', hue='Anemia', bins=30, kde=True, palette=['#2F5496', '#C00000'], ax=axes[1])
axes[1].axvline(11.0, color='black', linestyle='--', label='Corte MINSA (11.0 g/dL)')
axes[1].set_title('Distribución de Hemoglobina Ajustada')
axes[1].set_xlabel('Hemoglobina Ajustada por Altitud (g/dL)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Partición de Datos y Prevención Estricta de Data Leakage
Como se enseñó en clase (`cuaderno-04` y `cuaderno-07`):
1. **División Estratificada (80% Train, 20% Test):** Se conserva exactamente la proporción de anemia en ambas particiones.
2. **Ajuste Aislado:** `StandardScaler` se ajusta (`fit_transform`) **únicamente con X_train**. A `X_test` solo se le aplica `transform` para garantizar una evaluación ciega e incontaminada.

In [ ]:
features = [
    'Hemoglobina_Observada',
    'Altitud_msnm',
    'Edad_Meses',
    'Sexo',
    'Area_Residencia',
    'Peso_kg',
    'Talla_cm'
]

X = df[features]
y = df['Anemia']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f":: Muestras en Entrenamiento: {X_train.shape[0]}")
print(f":: Muestras en Prueba: {X_test.shape[0]}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(":: Escalado completado sin fuga de datos (Data Leakage prevenido).")

## 5. Optimización del Hiperparámetro k en KNN
Exploramos valores impares de $k \in [1, 23]$ sobre la población infantil peruana para hallar el punto óptimo entre sesgo y varianza.

In [ ]:
k_values = list(range(1, 25, 2))
train_scores = []
test_scores = []
f1_scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
    knn.fit(X_train_scaled, y_train)
    pred_te = knn.predict(X_test_scaled)
    train_scores.append(accuracy_score(y_train, knn.predict(X_train_scaled)))
    test_scores.append(accuracy_score(y_test, pred_te))
    f1_scores.append(f1_score(y_test, pred_te))

mejor_k = k_values[np.argmax(f1_scores)]
print(f":: Mejor k seleccionado para la población peruana: k = {mejor_k} (F1-Score = {max(f1_scores):.4f})")

plt.figure(figsize=(8, 4.5))
plt.plot(k_values, train_scores, marker='o', label='Exactitud Train', color='#2F5496')
plt.plot(k_values, test_scores, marker='s', label='Exactitud Test', color='#C00000')
plt.plot(k_values, f1_scores, marker='^', linestyle='--', label='F1-Score Test', color='#548235')
plt.title('Compromiso Sesgo-Varianza en KNN (ENDES Perú)')
plt.xlabel('Número de Vecinos (k)')
plt.ylabel('Puntuación')
plt.xticks(k_values)
plt.legend()
plt.tight_layout()
plt.show()

## 6. Entrenamiento y Comparativa Multicriterio de Modelos
Siguiendo la pauta del profesor, comparamos:
1. **Baseline Trivial (Dummy):** Predice siempre la clase mayoritaria.
2. **Baseline Lineal (Regresión Logística):** Frontera hiperplana.
3. **K-Nearest Neighbors (KNN):** Clasificador métrico por similitud de casos.
4. **Random Forest:** Ensamble de 100 árboles de decisión.

In [ ]:
modelos = {
    "Baseline (Dummy)": DummyClassifier(strategy="most_frequent"),
    "Regresión Logística": LogisticRegression(random_state=42, max_iter=1000),
    f"K-Nearest Neighbors (k={mejor_k})": KNeighborsClassifier(n_neighbors=mejor_k),
    "Random Forest (100 árboles)": RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42)
}

tabla = []
modelos_entrenados = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for nombre, mod in modelos.items():
    mod.fit(X_train_scaled, y_train)
    modelos_entrenados[nombre] = mod
    y_pred = mod.predict(X_test_scaled)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    
    if hasattr(mod, "predict_proba"):
        auc = roc_auc_score(y_test, mod.predict_proba(X_test_scaled)[:, 1])
    else:
        auc = 0.50
        
    cv_f1 = cross_val_score(mod, X_train_scaled, y_train, cv=cv, scoring='f1')
    
    tabla.append({
        "Modelo": nombre,
        "Accuracy": f"{acc*100:.2f}%",
        "Precision": f"{prec*100:.2f}%",
        "Recall (Sensibilidad)": f"{rec*100:.2f}%",
        "F1-Score": f"{f1:.4f}",
        "ROC-AUC": f"{auc:.4f}",
        "5-Fold CV F1": f"{cv_f1.mean():.4f} (± {cv_f1.std():.4f})"
    })

df_eval = pd.DataFrame(tabla)
display(df_eval)

## 7. Matrices de Confusión y Curvas ROC

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Matriz KNN
cm_knn = confusion_matrix(y_test, modelos_entrenados[f"K-Nearest Neighbors (k={mejor_k})"].predict(X_test_scaled))
disp_knn = ConfusionMatrixDisplay(cm_knn, display_labels=['No Anemia', 'Anemia'])
disp_knn.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title(f'Matriz de Confusión - KNN (k={mejor_k})')

# Matriz Random Forest
cm_rf = confusion_matrix(y_test, modelos_entrenados["Random Forest (100 árboles)"].predict(X_test_scaled))
disp_rf = ConfusionMatrixDisplay(cm_rf, display_labels=['No Anemia', 'Anemia'])
disp_rf.plot(ax=axes[1], cmap='Greens', colorbar=False)
axes[1].set_title('Matriz de Confusión - Random Forest')

plt.tight_layout()
plt.show()

In [ ]:
# Curvas ROC
plt.figure(figsize=(7, 5))
for mod_nom in [f"K-Nearest Neighbors (k={mejor_k})", "Random Forest (100 árboles)", "Regresión Logística"]:
    m = modelos_entrenados[mod_nom]
    y_prob = m.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, label=f"{mod_nom} (AUC = {auc:.3f})")

plt.plot([0, 1], [0, 1], 'k--', label='Clasificador Aleatorio (AUC = 0.500)')
plt.xlabel('Tasa de Falsos Positivos')
plt.ylabel('Tasa de Verdaderos Positivos (Recall)')
plt.title('Curvas ROC - Comparación en Población Infantil Peruana')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 8. Interpretabilidad: Importancia de Características y el Efecto de la Altitud
Analizamos qué peso asignó Random Forest a cada variable clínica y geográfica del Perú.

In [ ]:
rf_final = modelos_entrenados["Random Forest (100 árboles)"]
imp = rf_final.feature_importances_
cols = X.columns
idx = np.argsort(imp)[::-1]

plt.figure(figsize=(8, 4.5))
plt.bar(range(len(imp)), imp[idx], color='#2F5496')
plt.xticks(range(len(imp)), [cols[i] for i in idx], rotation=30, ha='right')
plt.title('Importancia de Variables Clínicas y Geográficas (Gini Importance)')
plt.ylabel('Importancia Relativa')
plt.tight_layout()
plt.show()

for i in idx:
    print(f"{cols[i]:<25} -> {imp[i]*100:.2f}%")

## 9. Prototipo de Diagnóstico Clínico en Tiempo Real
Serializamos los modelos e implementamos una función para que un médico o enfermero SERUMS en cualquier posta del Perú ingrese los datos de un niño y obtenga el resultado al instante.

In [ ]:
# Exportar artefactos .joblib
os.makedirs("models", exist_ok=True)
joblib.dump(modelos_entrenados["Random Forest (100 árboles)"], "models/modelo_anemia_rf.joblib")
joblib.dump(modelos_entrenados[f"K-Nearest Neighbors (k={mejor_k})"], "models/modelo_anemia_knn.joblib")
joblib.dump(scaler, "models/scaler_peru.joblib")
print(":: Modelos exportados exitosamente.")

def evaluar_nino_peruano(hemoglobina_obs, altitud_msnm, edad_meses, sexo, area_rural, peso_kg, talla_cm):
    """
    sexo: 1 (Niño / Varón), 0 (Niña / Mujer)
    area_rural: 1 (Zona Rural), 0 (Zona Urbana)
    """
    datos = np.array([[hemoglobina_obs, altitud_msnm, edad_meses, sexo, area_rural, peso_kg, talla_cm]])
    datos_esc = scaler.transform(datos)
    
    modelo = modelos_entrenados["Random Forest (100 árboles)"]
    pred = modelo.predict(datos_esc)[0]
    prob = modelo.predict_proba(datos_esc)[0][1] * 100
    
    diagnostico = "ANEMIA DETECTADA (ALERTA SANITARIA)" if pred == 1 else "SIN ANEMIA (VALORES NORMALES)"
    print("\n================ RESULTADO DEL TAMIZAJE (ENDES PERÚ) ================")
    print(f"-> Diagnóstico Clínico: {diagnostico}")
    print(f"-> Probabilidad de Anemia: {prob:.2f}%")
    print(f"-> Altitud considerada: {altitud_msnm} msnm")
    return pred, prob

# Caso 1: Niño de Puno (3,825 msnm) con Hb observada aparentemente 'normal' (12.0 g/dL)
# En Puno a 3,825 msnm, la altitud resta casi 3 g/dL, por lo que 12.0 es en realidad < 11.0 g/dL (TIENE ANEMIA)
print("--- CASO 1: Niño evaluado en posta de Puno (3,825 msnm) ---")
evaluar_nino_peruano(hemoglobina_obs=12.0, altitud_msnm=3825, edad_meses=18, sexo=1, area_rural=1, peso_kg=10.2, talla_cm=78.5)

# Caso 2: Niña de Lima (150 msnm) con Hb de 12.0 g/dL (SIN ANEMIA)
print("\n--- CASO 2: Niña evaluada en centro de salud de Lima (150 msnm) ---")
evaluar_nino_peruano(hemoglobina_obs=12.0, altitud_msnm=150, edad_meses=24, sexo=0, area_rural=0, peso_kg=12.0, talla_cm=86.0)